# Skillra PDA: HSE PDA Course project

Воспроизводимый аналитический отчёт Skillra о рынке IT-вакансий (в рамках ТЗ по PDA, этапы 0-4) и продуктовая часть с персонами и skill-gap.

## Вводная
Skillra решает проблему информационного шума на рынке IT-вакансий: кандидаты и джуны тратят недели на поиск релевантных ролей,
а работодатели плохо видят пул талантов и разрывы в навыках. Мы строим Career & Job Market Navigator, опираясь на сырые данные
hh.ru, чтобы проверять продуктовые гипотезы: какие сегменты растут, где есть junior-friendly окна, какие навыки дают премию.

Этот ноутбук - investor story и отчёт по проекту HSE PDA: этапы 0-4 показывают полный цикл от парсинга до визуализаций и персон Skillra.
Каждый блок заканчивается выводами: что мы сделали, зачем, что узнали про рынок и как это конвертируется в ценность продукта.

### Команда и набор задач
| Участник                                    | Зоны ответственности                                                                                                |
|---------------------------------------------|---------------------------------------------------------------------------------------------------------------------|
| Монахов Иван | Парсер hh.ru (`parser/hh_scraper.py`), ежедневный сбор данных, обход лимитов hh.ru, документация и быстрая отладка.                                |
| Адамов Даниил, Полынская Галина                                | Предобработка (`cleaning.py`), контроль булевых фич и зарплатных инвариантов, тесты. |
| Полынская Галина, Адамов Даниил             | Фичи (`features.py`), витрина рынка (`market.py`), интеграция путей/конфига, пайплайн (`scripts/run_pipeline.py`).  |
| Попов Контантин, Монахов Иван               | EDA и визуализации (`eda.py`, `viz.py`), метрики по ролям/городам/форматам, улучшение графиков и HTML-отчёта.       |
| Монахов Иван, Попов Константин              | Оркестрация шагов плана, чек-лист, поддержка notebook-runner и воспроизводимости, общее ревью решения.              |

### Настройка окружения и загрузка артефактов
Используем функции из `src.skillra_pda`, директории и пути из `config`. Если обработанные данные отсутствуют, запускаем пайплайн `scripts/run_pipeline.py`.

## Этап 0. Парсинг hh.ru и исходные данные
Парсер `parser/hh_scraper.py` ежедневно собирает активные IT-вакансии hh.ru по СНГ. Цель - накопить >1kk строк до конца 2026 года, чтобы видеть динамику рынка.
Сбор учитывает лимиты hh.ru (пагинация по опыту `EXPERIENCE_SHARDS`, ограничение ~2000 результатов на выдачу, дросселирование запросов и ротацию proxy/UA).

### 0.1 Что собирает парсер и как данные используются дальше
- Фильтры: широкая булева строка по IT-ролям (`DEFAULT_QUERY`), регионы СНГ (`areas`), опытные срезы (`EXPERIENCE_SHARDS`), лимит `DEFAULT_LIMIT`.
- Периодичность: ежедневный запуск (дельта активных вакансий), цель >1kk строк c заданными заработными платами.
- Ограничения hh.ru: лимит на выдачу, антибот, необходимость пауз и ротации User-Agent/proxy.
- Куда идёт дальше: поля из CSV мапятся на признаки из `docs/02_feature_dictionary_hh.md` (формат работы, грейд, роли, стек, бенефиты, англ/образование) и используются в пайплайне + продуктовых персонах.

In [ ]:

from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import Markdown, display, Image

CWD = Path.cwd()
CANDIDATES = [CWD] + list(CWD.parents)
PROJECT_ROOT = next((p for p in CANDIDATES if (p / "src" / "skillra_pda").exists()), CWD)
SRC_DIR = PROJECT_ROOT / "src"

for path in {PROJECT_ROOT, SRC_DIR}:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from src.skillra_pda import config, io, cleaning, features, eda, viz, personas
from importlib import reload
for module in (cleaning, features, eda):
    reload(module)

config.ensure_directories()

if not config.CLEAN_DATA_FILE.exists() or not config.FEATURE_DATA_FILE.exists():
    import scripts.run_pipeline as run_pipeline
    run_pipeline.main()

raw_path = config.RAW_DATA_FILE
clean_path = config.CLEAN_DATA_FILE
feat_path = config.FEATURE_DATA_FILE

df_raw = io.load_raw(raw_path)
df_clean = pd.read_parquet(clean_path)
df_features = pd.read_parquet(feat_path)

grade_df = df_features.copy()
grade_df["grade"] = grade_df["grade_final"]

df_features.head(2)


### Эволюция датасета
Прослеживаем размеры и ключевые доли на каждом этапе пайплайна: сырой CSV → очищенный parquet → фичи → агрегированная витрина `market_view`.

In [ ]:

from src.skillra_pda import market

market_view_path = config.PROCESSED_LATEST_DIR / "market_view.parquet"
df_market_view = (
    pd.read_parquet(market_view_path)
    if market_view_path.exists()
    else market.build_market_view(df_features.copy())
)

def summarize_stage(name: str, df, weight_col: str | None = None):
    rows, cols = df.shape
    salary_fields = [c for c in ["salary_from", "salary_to", "salary_mid"] if c in df]
    salary_range_specified_share = (
        df[salary_fields].notna().any(axis=1).mean() if salary_fields else None
    )
    salary_rub_available_share = (
        df["salary_mid_rub_capped"].notna().mean() if "salary_mid_rub_capped" in df else None
    )

    currency_series = df.get("currency")
    usd_share = eur_share = unknown_currency_share = non_rub_share = None
    if currency_series is not None:
        currency = currency_series.fillna("unknown").str.upper()
        usd_share = currency.eq("USD").mean()
        eur_share = currency.eq("EUR").mean()
        unknown_currency_share = currency.eq("UNKNOWN").mean()
        non_rub_share = 1 - currency.eq("RUB").mean()

    remote_share = None
    unknown_work = None
    if "work_mode" in df:
        work_mode_series = df["work_mode"].fillna("unknown").str.lower()
        remote_share = work_mode_series.isin(["remote", "hybrid"]).mean()
        unknown_work = work_mode_series.eq("unknown").mean()
    elif "is_remote" in df:
        remote_share = df["is_remote"].fillna(False).mean()
    elif "remote_share" in df:
        weights = df[weight_col] if weight_col and weight_col in df else None
        if weights is not None and weights.sum() > 0:
            remote_share = float((df["remote_share"] * weights).sum() / weights.sum())
        else:
            remote_share = df["remote_share"].mean()

    return {
        "stage": name,
        "rows": rows,
        "cols": cols,
        "salary_range_specified_share": salary_range_specified_share,
        "salary_rub_available_share": salary_rub_available_share,
        "non_rub_share": non_rub_share,
        "usd_share": usd_share,
        "eur_share": eur_share,
        "unknown_currency_share": unknown_currency_share,
        "remote_or_hybrid_share": remote_share,
        "unknown_work_mode_share": unknown_work,
    }

evolution_rows = [
    summarize_stage("raw", df_raw),
    summarize_stage("clean", df_clean),
    summarize_stage("features", df_features),
    summarize_stage("market_view", df_market_view, weight_col="vacancy_count_total"),
]
evolution_df = pd.DataFrame(evolution_rows)
display(evolution_df)


In [ ]:

from IPython.display import Markdown, display

evo = evolution_df.set_index("stage")
raw_stage = evo.loc["raw"]
clean_stage = evo.loc["clean"]
feat_stage = evo.loc["features"]
market_stage = evo.loc["market_view"]

non_rub_text = (
    "N/A (валюта унифицирована в features → market_view хранит только RUB)"
    if pd.isna(raw_stage.get("non_rub_share"))
    else (
        f"{raw_stage['non_rub_share']:.1%} (USD {raw_stage['usd_share']:.1%}, "
        f"EUR {raw_stage['eur_share']:.1%}, unknown {raw_stage['unknown_currency_share']:.1%})"
    )
)

salary_range_fields = "salary_from/salary_to/salary_mid"

def fmt_int(x: int) -> str:
    return f"{int(x):,}".replace(",", " ")

def fmt_pct(value):
    return "N/A" if value is None or pd.isna(value) else f"{value:.1%}"

summary_md = Markdown(
    f"**Ключевые выводы по эволюции:** "
    f"n {fmt_int(raw_stage['rows'])} → {fmt_int(feat_stage['rows'])}, "
    f"колонок {fmt_int(raw_stage['cols'])} → {fmt_int(feat_stage['cols'])}; "
    f"salary_range_specified_share ({salary_range_fields}) после clean - {fmt_pct(clean_stage['salary_range_specified_share'])}; "
    f"salary_rub_available_share (salary_mid_rub_capped) на features - {fmt_pct(feat_stage['salary_rub_available_share'])}; "
    f"non-RUB на raw - {non_rub_text}; "
    f"remote+hybrid на этапе features - {fmt_pct(feat_stage['remote_or_hybrid_share'])}, "
    f"work_mode=unknown - {fmt_pct(feat_stage['unknown_work_mode_share'])}."
)
display(summary_md)


### 0.1a Здоровье данных: NaN и маркеры unknown
Проверяем, где остаются пропуски и текстовые маркеры неопределённости на этапах raw → clean → features.

In [ ]:
health_raw = cleaning.summarize_data_health(df_raw, prefix="raw:")
health_clean = cleaning.summarize_data_health(df_clean, prefix="clean:")
health_features = cleaning.summarize_data_health(df_features, prefix="feat:")

def top_health(df: pd.DataFrame, n: int = 12):
    return df.sort_values(by=["share_nan", "share_unknown_marker"], ascending=False).head(n)

health_raw_top = top_health(health_raw)
health_clean_top = top_health(health_clean)
health_features_top = top_health(health_features)

display(health_raw_top), display(health_clean_top), display(health_features_top)

In [ ]:

LOW_INFO_THRESHOLD = cleaning.LOW_INFORMATION_THRESHOLD
low_info_raw = cleaning.list_low_information_columns(df_raw, threshold=LOW_INFO_THRESHOLD)
low_info_raw = low_info_raw.assign(
    in_clean=low_info_raw["column"].isin(df_clean.columns)
)
low_info_raw["decision"] = low_info_raw["in_clean"].map(
    {True: "оставили (whitelist/аналитическая ценность)", False: "удалили при cleaning"}
)
low_info_raw_head = low_info_raw.head(20)
low_info_raw_head


In [ ]:
COVERAGE_ALERT_THRESHOLD = 0.15
coverage_columns = [
    "column",
    "coverage_share",
    "effective_missing_share",
    "quality_bucket",
    "recommended_use",
    "comment",
]

low_coverage_clean = (
    health_clean[health_clean["coverage_share"] < COVERAGE_ALERT_THRESHOLD]
    .sort_values(by="coverage_share")
    .head(12)
)

display(low_coverage_clean[coverage_columns])


def _fmt_cols(series, max_items: int = 8) -> str:
    cols = series.tolist()
    if not cols:
        return '-'
    head = ', '.join(cols[:max_items])
    if len(cols) > max_items:
        head += ', …'
    return head


policy_md = Markdown(
    f"""
**Политика использования признаков по coverage:**
- >=85% (high) - ок для ключевых выводов: {_fmt_cols(health_clean[health_clean['quality_bucket'] == 'high']['column'])}.
- 60–85% (medium) - используем с дисклеймером: {_fmt_cols(health_clean[health_clean['quality_bucket'] == 'medium']['column'])}.
- 30–60% (low) - формируем только гипотезы: {_fmt_cols(health_clean[health_clean['quality_bucket'] == 'low']['column'])}.
- <30% (critical) - игнорируем в твёрдых выводах: {_fmt_cols(health_clean[health_clean['quality_bucket'] == 'critical']['column'])}.
"""
)

display(policy_md)

#### Low-information columns и покрытие
Колонки с `effective_missing_share ≥ 90%` считаем шумовыми и удаляем при cleaning. Поля с низким coverage (<15%) в таблице выше не используем для твёрдых выводов: они годятся только для гипотез/обогащения, а whitelist удерживает критичные признаки, чтобы показывать их с явным дисклеймером.

In [ ]:
default_obj_clean = pd.Series(pd.NA, index=df_clean.index, dtype=object)
default_obj_feat = pd.Series(pd.NA, index=df_features.index, dtype=object)
default_bool_clean = pd.Series(pd.NA, index=df_clean.index, dtype="boolean")

def _share_unknown(series):
    if series is None:
        return None
    return (
        pd.Series(series)
        .fillna("unknown")
        .astype(str)
        .str.lower()
        .eq("unknown")
        .mean()
    )

work_mode_unknown_share = _share_unknown(df_features.get("work_mode", default_obj_feat))
english_unknown_share = _share_unknown(df_clean.get("lang_english_level", default_obj_clean))
english_required_nan = df_clean.get("lang_english_required", default_bool_clean).isna().mean()
edu_source = df_clean.get("edu_level") if "edu_level" in df_clean else df_features.get("edu_level")
education_unknown_share = _share_unknown(edu_source)
benefits_missing_share = df_clean.filter(like="benefit_").isna().mean().mean() if df_clean.filter(like="benefit_").shape[1] else 0
soft_missing_share = df_clean.filter(like="soft_").isna().mean().mean() if df_clean.filter(like="soft_").shape[1] else 0

grade_unknown_raw = _share_unknown(df_clean.get("grade", default_obj_clean))
grade_unknown_final = _share_unknown(df_features.get("grade_final", default_obj_feat))
salary_mid = df_features.get("salary_mid_rub")
salary_capped = df_features.get("salary_mid_rub_capped")
salary_capped_share = (
    (salary_mid.notna()) & (salary_capped.notna()) & (salary_mid != salary_capped)
).mean() if salary_mid is not None and salary_capped is not None else 0

def _fmt(value):
    if value is None or pd.isna(value):
        return "N/A"
    return f"{value:.1%}"

health_key_figures = {
    "work_mode_unknown_share": work_mode_unknown_share,
    "english_unknown_share": english_unknown_share,
    "english_required_nan": english_required_nan,
    "education_unknown_share": education_unknown_share,
    "benefits_missing_share": benefits_missing_share,
    "soft_missing_share": soft_missing_share,
    "grade_unknown_raw": grade_unknown_raw,
    "grade_unknown_final": grade_unknown_final,
    "salary_capped_share": salary_capped_share,
}

health_key_figures_fmt = {k: _fmt(v) for k, v in health_key_figures.items()}
display(health_key_figures_fmt)


In [ ]:
from IPython.display import Markdown, display

n_features = len(df_features)
grade_series = df_features.get("grade_final", df_features.get("grade", pd.Series(dtype=str)))
grade_counts = grade_series.fillna("unknown").astype(str).str.lower().value_counts()
work_mode_counts = df_features.get("work_mode", pd.Series(dtype=str)).fillna("unknown").astype(str).str.lower().value_counts()
english_unknown = (
    df_features.get("lang_english_level", pd.Series(dtype=str))
    .fillna("unknown")
    .astype(str)
    .str.lower()
    .eq("unknown")
    .mean()
)
salary_missing = df_features[["salary_from", "salary_to"]].isna().all(axis=1).mean()

grade_unknown_share = grade_counts.get("unknown", 0) / max(n_features, 1)
work_mode_unknown_share = work_mode_counts.get("unknown", 0) / max(n_features, 1)

coverage_md = Markdown(
    f"**Coverage ключевых полей:** выборка {n_features:,}. "
    f"unknown grade - {grade_unknown_share:.1%}, "
    f"unknown work_mode - {work_mode_unknown_share:.1%}, "
    f"unknown уровень английского - {english_unknown:.1%}, "
    f"пропусков зарплатной вилки - {salary_missing:.1%}. "
    f"Высокая доля неизвестных грейдов и форматов работы может смещать распределения."
)
display(coverage_md)


In [ ]:
data_quality_table = pd.DataFrame(
    [
        (
            "Work mode = unknown",
            f"{work_mode_unknown_share:.1%}",
            "Доля удалёнки/гибрида может быть занижена; в выводах отмечаем неопределённость формата.",
        ),
        (
            "English level unknown",
            f"{english_unknown_share:.1%}",
            "Оценка требований к языку ограничена, считаем доли среди известных значений.",
        ),
        (
            "Grade unknown (raw) → grade_final",
            f"{grade_unknown_raw:.1%} → {grade_unknown_final:.1%}",
            "grade_final сокращает неопределённость, графики строим по итоговому грейду.",
        ),
        (
            "Salary cap (capped vs mid)",
            f"{salary_capped_share:.1%}",
            "Срезали верхние выбросы, поэтому медианы/топы не раздуваются редкими офферами.",
        ),
        (
            "Benefits missing",
            f"{benefits_missing_share:.1%}",
            "Интерпретируем доли бенефитов с оговоркой о пропусках.",
        ),
        (
            "Soft skills missing",
            f"{soft_missing_share:.1%}",
            "Топ soft-skills строим по имеющимся строкам, не обобщая на весь рынок.",
        ),
    ],
    columns=["Метрика", "Значение", "Как влияет на выводы"],
)

display(data_quality_table)

quality_md = Markdown(
    f"""
**Что увидели по качеству:**
- Work_mode остаётся слабым местом: {work_mode_unknown_share:.1%} строк с unknown → в сегментациях по формату работы делаем явную оговорку.
- Требования к английскому указаны эпизодически (unknown {english_unknown_share:.1%} + {english_required_nan:.1%} NaN по "обязательности"), поэтому выводы по языку строим среди известных значений.
- grade_final сокращает долю unknown грейда с {grade_unknown_raw:.1%} до {grade_unknown_final:.1%}, что делает графики по грейдам стабильнее.
- Salary cap применился к {salary_capped_share:.1%} строк: медианы устойчивы, но высокие офферы не завышают рынок.
"""
)

display(quality_md)


### 0.2 Пример сырых данных и базовые статы
Показываем head, `info()` и первичный профиль пропусков/типов, чтобы понимать качество источника до очистки.

In [ ]:

import io

raw_head = df_raw.head(5)
buffer = io.StringIO()
df_raw.info(buf=buffer)
raw_info = buffer.getvalue()

raw_overview = {
    "raw_path": str(raw_path),
    "raw_shape": df_raw.shape,
    "columns_sample": df_raw.columns.tolist()[:10],
}
raw_profile = cleaning.basic_profile(df_raw)
raw_missing_top = eda.missing_share(df_raw, top_n=10)
raw_numeric_stats = df_raw.select_dtypes("number").describe().T

display(raw_head)
print(raw_info)
display(pd.Series(raw_overview))
display(raw_profile)
display(raw_missing_top)
display(raw_numeric_stats)


In [ ]:
from IPython.display import Markdown, display

raw_rows, raw_cols = raw_stage["rows"], raw_stage["cols"]
raw_salary_share = raw_stage.get("salary_specified_share")
raw_non_rub = raw_stage.get("non_rub_share")

pct = lambda v: "N/A" if v is None or pd.isna(v) else f"{v:.1%}"
raw_non_rub_text = pct(raw_non_rub)

raw_md = Markdown(
    f"""### 0.3 Выводы по этапу 0
- **Что сделали:** описали ежедневный парсер hh.ru, его фильтры/ограничения, показали сырой head и профиль данных.
- **Почему так:** ежедневная дельта и лимиты hh.ru требуют шардирования опыта и аккуратных пауз, чтобы собирать стабильный поток вакансий.
- **Ключевые цифры:** {fmt_int(raw_rows)} строк и {fmt_int(raw_cols)} колонок в raw; {pct(raw_salary_share)} строк содержат зарплатную вилку; non-RUB - {raw_non_rub_text}.
- **Мини-чек-лист:** после парсинга имеем сырую выгрузку {fmt_int(raw_rows)}×{fmt_int(raw_cols)} без дублей; зафиксировали поля зарплаты/валюты как основу для очистки и конвертации.
- **Что это даёт Skillra:** стабильная точка входа данных для аналитики и рекомендации карьерных путей, масштабируемая до 1kk+ вакансий.
"""
)

linked_md = Markdown("Следующий шаг - очистка и унификация зарплат/валют, чтобы EDA не искажался.")
display(raw_md, linked_md)


## Этап 1. Предобработка (cleaning.py)
Повторяем шаги cleaning: парсинг дат, дедупликация, обработка пропусков, нормализация булевых колонок (`salary_gross` и все флаги is_/has_/benefit_/soft_/domain_/role_). Готовые результаты берём из `data/processed/latest/hh_clean.parquet`. Ниже - снимки пропусков и типов.

In [ ]:

dup_raw = df_raw.duplicated().sum()
dup_clean = df_clean.duplicated().sum()
missing_salary = df_raw[[c for c in df_raw.columns if "salary" in c]].isna().mean().to_dict()

clean_missing_top = eda.missing_share(df_clean, top_n=10)
salary_gross_dtype = str(df_clean.get("salary_gross", pd.Series(dtype="boolean")).dtype)
salary_gross_stats = df_clean.get("salary_gross", pd.Series(dtype="boolean")).value_counts(dropna=False)

clean_snapshot = {
    "clean_shape": df_clean.shape,
    "salary_gross_dtype": salary_gross_dtype,
    "salary_gross_counts": salary_gross_stats.to_dict(),
    "duplicates_raw": int(dup_raw),
    "duplicates_clean": int(dup_clean),
    "missing_salary_share_raw": missing_salary,
}

display(pd.DataFrame([clean_snapshot]))
display(clean_missing_top)


In [ ]:
from IPython.display import Markdown, display

clean_rows, clean_cols = clean_stage["rows"], clean_stage["cols"]
clean_salary_from = df_clean.get("salary_from").notna().mean() if "salary_from" in df_clean else None
clean_salary_to = df_clean.get("salary_to").notna().mean() if "salary_to" in df_clean else None
pct = lambda v: "N/A" if v is None or pd.isna(v) else f"{v:.1%}"

clean_md = Markdown(
    f"""### 1.1 Выводы по этапу 1
- **Что сделали:** привели типы, очистили пропуски, дедуплицировали, нормализовали зарплаты и каппинг.
- **Почему так:** без единых типов и контроля `salary_gross`/валюты нельзя честно сравнивать роли и города.
- **Ключевые цифры:** размер датасета сохранился {fmt_int(clean_rows)}×{fmt_int(clean_cols)}, дублей нет; `salary_from` заполнен в {pct(clean_salary_from)} строк, `salary_to` - в {pct(clean_salary_to)}.
- **Мини-чек-лист:** raw → clean без потери строк ({fmt_int(clean_rows)}×{fmt_int(clean_cols)}), конвертировали валюту и подготовили `salary_mid_rub_capped`.
- **Что это даёт Skillra:** очищенные данные → стабильные витрины и корректные графики/персоны без шумовых выбросов.
"""
)

display(clean_md)


## Этап 2. Признаки (features.py)
Используем `features.engineer_all_features`: свежесть вакансии (`vacancy_age_days`), география (`city_tier`), формат работы (`work_mode`), зарплатные бакеты (`salary_bucket`), стековые агрегаты (`core_data_skills_count`, `ml_stack_count`, `tech_stack_size`), текстовые метрики (`description_len_chars/words`), продуктовые флаги (`is_junior_friendly`, `battle_experience`).

In [ ]:

key_features = [
    "vacancy_age_days",
    "city_tier",
    "work_mode",
    "salary_bucket",
    "core_data_skills_count",
    "ml_stack_count",
    "tech_stack_size",
    "description_len_chars",
    "description_len_words",
    "is_junior_friendly",
    "battle_experience",
]
available = [c for c in key_features if c in df_features.columns]

feature_snapshot = df_features[available].head(5)
feature_stats = df_features[[c for c in available if pd.api.types.is_numeric_dtype(df_features[c])]].describe().T

display(feature_snapshot)
display(feature_stats)


In [ ]:
from IPython.display import Markdown, display

feat_rows, feat_cols = feat_stage["rows"], feat_stage["cols"]
remote_share = feat_stage.get("remote_or_hybrid_share")
unknown_work_mode = feat_stage.get("unknown_work_mode_share")

pct = lambda v: "N/A" if v is None or pd.isna(v) else f"{v:.1%}"
office_field = None
if remote_share is not None and unknown_work_mode is not None and not pd.isna(remote_share) and not pd.isna(unknown_work_mode):
    office_field = max(0.0, 1 - remote_share - unknown_work_mode)

feature_md = Markdown(
    f"""### 2.1 Выводы по этапу 2
- **Что сделали:** добавили признаки work_mode, city_tier, salary_bucket, домены, роли, размер стека, junior-friendly, английский/образование.
- **Почему так:** эти фичи нужны для продукта (подбор ролей/треков) и для честного EDA по сегментам.
- **Ключевые цифры:** после feature engineering {fmt_int(feat_rows)}×{fmt_int(feat_cols)}; remote/hybrid - {pct(remote_share)}, офис/field - {pct(office_field)}, `unknown` work_mode - {pct(unknown_work_mode)}.
- **Мини-чек-лист:** до этапа {fmt_int(clean_stage['rows'])}×{fmt_int(clean_stage['cols'])} clean → после {fmt_int(feat_rows)}×{fmt_int(feat_cols)} features; добавили work_mode, city_tier, salary_bucket, стековые счётчики, primary_role, флаги junior-friendly/remote.
- **Что это даёт Skillra:** готовая витрина для подсказок пользователю и расчёта skill-gap по выбранному сегменту.
"""
)

transition_md = Markdown("Далее используем эту витрину в EDA и визуализациях, чтобы проверить продуктовые гипотезы.")
display(feature_md, transition_md)


## Этап 3. Разведочный анализ (EDA)
Смотрим рынок по зарплатам, форматам работы, ролям, навыкам, доменам, английскому, образованию и работодателям. Используем агрегаты из `eda.py` и визуализации из `viz.py`; добавляем текстовые выводы для каждой темы.

In [ ]:

import importlib

import src.skillra_pda.eda as eda_module

eda = importlib.reload(eda_module)
if not hasattr(eda, "salary_by_city_tier"):
    raise AttributeError("expected salary_by_city_tier in eda module; ensure local code is on sys.path")

salary_city = eda.salary_by_city_tier(df_features)
salary_grade = eda.salary_by_grade(grade_df)
salary_role = eda.salary_by_primary_role(df_features)
salary_stack = eda.salary_by_stack_size(df_features)

skill_cols = eda.hard_skill_columns(df_features)
top_skills = eda.skill_frequency(df_features, skill_cols, top_n=15).index.tolist()
skill_share = eda.skill_share_by_grade(grade_df, top_skills)

benefits_grade = eda.benefits_summary_by_grade(grade_df)
soft_corr = eda.soft_skills_correlation(df_features)
junior_roles = eda.junior_friendly_share(df_features, group_col="primary_role")

display(salary_city.head())
display(salary_grade.head())
display(salary_role.head())
display(salary_stack.head())
display(skill_share.head())
display(benefits_grade.head())
display(soft_corr.head())
display(junior_roles.head())


### Быстрые sanity-checks (мягкие предупреждения)

In [ ]:
from IPython.display import Markdown, display

def warn(text: str):
    print(f"WARNING: {text}")

city_col = "city_tier"
work_mode_col = "work_mode"
salary_col = "salary_mid_rub_capped"

if city_col in df_features.columns:
    city_distribution = df_features[city_col].value_counts(normalize=True, dropna=False).head(5)
    print("Топ-5 распределения city_tier:")
    print(city_distribution)
    moscow_spb_share = city_distribution.get("Moscow", 0) + city_distribution.get("SPb", 0)
else:
    warn("Колонка `city_tier` отсутствует: не можем посчитать распределение.")
    city_distribution = None
    moscow_spb_share = None

if work_mode_col in df_features.columns:
    work_mode_share = df_features[work_mode_col].value_counts(normalize=True, dropna=False)
    remote_hybrid_share = work_mode_share.get("remote", 0) + work_mode_share.get("hybrid", 0)
else:
    warn("Колонка `work_mode` отсутствует: не можем посчитать долю remote/hybrid.")
    remote_hybrid_share = None

salary_series = df_features.get(salary_col)
if salary_series is not None:
    salary_share = float(salary_series.notna().mean()) if len(salary_series) else 0
else:
    warn("Колонка `salary_mid_rub_capped` отсутствует: не можем посчитать долю зарплат.")
    salary_share = None

if remote_hybrid_share is not None and remote_hybrid_share == 0:
    warn("remote+hybrid_share = 0 (проверьте, что корректно пропаршен work_mode).")

if moscow_spb_share is not None and moscow_spb_share == 0:
    warn("moscow_spb_share = 0 (в распределении city_tier нет Москвы/СПб?).")

if salary_share is not None and salary_share == 0:
    warn("salary_share = 0 (зарплаты не заполнены).")


Sanity-checks проверяют три вещи:
- есть ли базовые поля `city_tier`, `work_mode`, `salary_mid_rub_capped` и не пустые ли доли ключевых категорий;
- не обнулилась ли доля remote/hybrid и Москва+СПб;
- есть ли в данных зарплаты.

`WARNING` в выводе означает, что источник данных неполный или доли упали до нуля. В таком случае метрики ниже читаем аккуратно: отмечаем возможное смещение по географии/формату работы и проговариваем, что зарплатные выводы носят оценочный характер.

### 3.1 Рынок по доменам (отраслям)
Используем доменные флаги (`domain_*`), чтобы увидеть распределение вакансий и зарплат по ключевым отраслям. Приоритеты строим по частоте, чтобы корректно выбрать главный домен при нескольких флагах.

In [ ]:
domain_cols = [c for c in df_features.columns if c.startswith("domain_")]
domain_priorities = (
    df_features[domain_cols]
    .sum()
    .sort_values(ascending=False)
    .index.str.removeprefix("domain_")
    .tolist()
)


In [ ]:
from src.skillra_pda import viz
viz = importlib.reload(viz)

salary_domain_path = viz.salary_by_domain_plot(
    df_features, top_n=10, priorities=domain_priorities, return_fig=False
)


In [ ]:
domain_salary = eda.describe_salary_by_domain(df_features, priorities=domain_priorities)
domain_salary.head(10)

In [ ]:

from IPython.display import Markdown, display

salary_col = "salary_mid_rub_capped"
salary_series = df_features.get(salary_col)
salary_notna = salary_series.notna() if salary_series is not None else pd.Series([], dtype=bool)
salary_count = int(salary_notna.sum())
salary_share = float(salary_notna.mean()) if len(salary_notna) else 0

main_domains = domain_salary.head(5)
unknown_share_series = domain_salary.loc[domain_salary['domain'] == 'unknown', 'share']
unknown_share = float(unknown_share_series.iloc[0]) if not unknown_share_series.empty else 0

count_col = 'vacancy_count_total' if 'vacancy_count_total' in domain_salary else 'vacancy_count'
market_n_total = int(domain_salary[count_col].sum()) if count_col in domain_salary else len(df_features)

domain_lines = [
    f"Финтех - {main_domains.iloc[1]['share']:.1%} выборки с медианой {main_domains.iloc[1]['salary_median']:.0f} ₽",
    f"Продуктовые IT - {main_domains.iloc[2]['share']:.1%} с медианой {main_domains.iloc[2]['salary_median']:.0f} ₽",
]

domain_md = Markdown(f"""
#### Выводы по доменам
n={market_n_total:,} вакансий (зарплата указана у {salary_count:,} ({salary_share:.1%})); топ-2 домена: {domain_lines[0]}, {domain_lines[1]}. Unknown домен - {unknown_share:.1%}, поэтому зарплаты по отраслям трактуем осторожно.
""")
display(domain_md)


### 3.2 Зарплаты и формат работы
Комбинируем грейд×город и роль×формат, чтобы увидеть премии за локацию и удалёнку.

In [ ]:

from IPython.display import Markdown, display

def _fmt_salary(value):
    return f"{value:,.0f}".replace(",", " ") if pd.notna(value) else "N/A"

grade_df_local = grade_df.copy()

grade_summary_local = eda.salary_summary_by_grade_and_city(grade_df_local)
grade_city_pivot_local = grade_summary_local.pivot(index="grade", columns="city_tier", values="median")

city_tier_share = df_features.get("city_tier", pd.Series(dtype=str)).value_counts(normalize=True)
moscow_spb_share = city_tier_share.get("Moscow", 0) + city_tier_share.get("SPb", 0)
million_share = city_tier_share.get("Million+", 0)

city_priority = [c for c in ["Moscow", "SPb"] if c in grade_city_pivot_local.columns]
if len(city_priority) < 2:
    for col in city_tier_share.index:
        if col in grade_city_pivot_local.columns and col not in city_priority:
            city_priority.append(col)
        if len(city_priority) >= 2:
            break
if not city_priority:
    city_priority = list(grade_city_pivot_local.columns[:2])

moscow_col = city_priority[0]
region_col = city_priority[1] if len(city_priority) > 1 else city_priority[0]

def _safe_get(row_label, col_label):
    try:
        return grade_city_pivot_local.loc[row_label, col_label]
    except Exception:
        return pd.NA

senior_moscow = _safe_get("senior", moscow_col)
senior_region = _safe_get("senior", region_col)
middle_moscow = _safe_get("middle", moscow_col)
middle_region = _safe_get("middle", region_col)
junior_moscow = _safe_get("junior", moscow_col)
junior_region = _safe_get("junior", region_col)

city_md = Markdown(
    f"""#### 3.2.1 Зарплаты по грейду и типу города
Медианы по грейду/локации: senior - {_fmt_salary(senior_moscow)} ₽ в {moscow_col} против {_fmt_salary(senior_region)} ₽ в {region_col}; middle - {_fmt_salary(middle_moscow)} ₽ против {_fmt_salary(middle_region)} ₽; стажёры держатся на {_fmt_salary(junior_moscow)} ₽ в {moscow_col} и {_fmt_salary(junior_region)} ₽ в {region_col}. География объёма: Москва/СПб дают {moscow_spb_share:.1%} выборки, миллионники - ещё {million_share:.1%}.
""",
)

display(city_md)


In [ ]:

grade_order = viz.GRADE_ORDER

grade_city_summary = eda.salary_summary_by_grade_and_city(grade_df)
grade_city_summary["grade"] = pd.Categorical(grade_city_summary["grade"], categories=grade_order, ordered=True)
grade_city_summary = grade_city_summary.sort_values(["grade", "city_tier"])
grade_city_path = viz.salary_by_grade_city_heatmap(
    grade_city_summary, return_fig=False
)


In [ ]:

grade_city_pivot = grade_city_summary.pivot(index="grade", columns="city_tier", values="median")
city_tier_share = df_features["city_tier"].value_counts(normalize=True)
salary_spread = {
    "senior_moscow": grade_city_pivot.loc["senior", "Moscow"],
    "senior_regions": grade_city_pivot.loc["senior", "Other RU"],
    "middle_moscow": grade_city_pivot.loc["middle", "Moscow"],
    "middle_regions": grade_city_pivot.loc["middle", "Other RU"],
    "intern_moscow": grade_city_pivot.loc["intern", "Moscow"],
}

display(city_tier_share)
display(pd.DataFrame([salary_spread]))


In [ ]:

from IPython.display import Markdown, display

def _fmt_salary(value):
    return f"{value:,.0f}".replace(",", " ") if pd.notna(value) else "N/A"

city_salary_table = pd.DataFrame(
    {
        "senior": {"Moscow": salary_spread.get("senior_moscow"), "Other RU": salary_spread.get("senior_regions")},
        "middle": {"Moscow": salary_spread.get("middle_moscow"), "Other RU": salary_spread.get("middle_regions")},
        "intern": {"Moscow": salary_spread.get("intern_moscow"), "Other RU": pd.NA},
    }
)
city_salary_table.index.name = "city_tier"
display(city_salary_table)

city_md = Markdown(
    f"""*График: тепловая карта зарплат по грейду и типу города.*

- Senior: Москва/СПб {_fmt_salary(salary_spread.get('senior_moscow'))} RUB против {_fmt_salary(salary_spread.get('senior_regions'))} RUB в регионах (премия {_fmt_salary(salary_spread.get('senior_moscow') - salary_spread.get('senior_regions'))} RUB).
- Middle: {_fmt_salary(salary_spread.get('middle_moscow'))} RUB в Москве и {_fmt_salary(salary_spread.get('middle_regions'))} RUB в регионах; на Москву приходится {city_tier_share.get('Moscow', 0):.1%} выборки.
- Intern медиана в Москве {_fmt_salary(salary_spread.get('intern_moscow'))} RUB; Other RU дают {city_tier_share.get('Other RU', 0):.1%} вакансий и задают нижний уровень вилки."""
)
display(city_md)


In [ ]:
from IPython.display import Markdown, display

city_tier_series = df_features.get("city_tier", pd.Series(dtype=str))
city_tier_share = city_tier_series.value_counts(normalize=True)
moscow_plus_million = (
    city_tier_share.get("Moscow", 0)
    + city_tier_share.get("SPb", 0)
    + city_tier_share.get("Million+", 0)
)

if city_tier_share.sum() > 0:
    city_line = (
        "На тепловых картах видно, что основные зарплатные объёмы сосредоточены в Москве и крупных городах "
        f"(≈{moscow_plus_million:.1%} вакансий), премия по грейду сохраняется даже при узких выборках."
    )
else:
    city_line = (
        "На тепловых картах видно основные объёмы, но в данных нет распределения city_tier - "
        "долю Москвы/крупных городов оценить нельзя."
    )

volume_md = Markdown(
    f"""#### 3.2.1a Дополнительные salary-срезы (объём + медиана)
{city_line}
""",
)

display(volume_md)


In [ ]:
salary_city_path = viz.salary_by_city_mean_count_plot(
    df_features, top_n=8, return_fig=False
)

display(Image(salary_city_path))


График: медианные зарплаты и объём вакансий по типу города.
- **Рынок:** Москва/1-й эшелон дают львиную долю спроса и более высокие медианы.
- **Skillra:** можно приоритизировать рекомендации по топ-городам и показывать разницу удалёнки vs офис.

In [ ]:
grade_for_bars = grade_df.copy()
salary_grade_counts_path = viz.salary_by_grade_mean_count_plot(
    grade_for_bars, return_fig=False
)

display(Image(salary_grade_counts_path))


In [ ]:

from IPython.display import Markdown, display

def _fmt_salary(value):
    return f"{value:,.0f}".replace(",", " ") if pd.notna(value) else "N/A"

grade_summary = (
    grade_for_bars.groupby("grade")["salary_mid_rub_capped"]
    .agg(vacancy_count_total="size", salary_median="median")
    .reindex(grade_order)
    .dropna(subset=["vacancy_count_total"])
)
display(grade_summary)

middle_row = grade_summary.loc["middle"]
senior_row = grade_summary.loc["senior"]
junior_rows = grade_summary.loc[[g for g in ["intern", "junior"] if g in grade_summary.index]]

grade_md = Markdown(
    f"""График: медианные зарплаты и объём вакансий по грейдам.

- Middle лидирует по спросу: {int(middle_row.vacancy_count_total)} вакансий с медианой {_fmt_salary(middle_row.salary_median)} RUB.
- Senior премия до {_fmt_salary(senior_row.salary_median)} RUB (≈{_fmt_salary(senior_row.salary_median - middle_row.salary_median)} RUB к middle) при {int(senior_row.vacancy_count_total)} вакансиях.
- Intern+junior суммарно {int(junior_rows.vacancy_count_total.sum())} строк; медиана junior {_fmt_salary(junior_rows.loc['junior', 'salary_median']) if 'junior' in junior_rows.index else 'N/A'} RUB задаёт уровень входа."""
)
display(grade_md)


In [ ]:
salary_role_counts_path = viz.salary_by_primary_role_mean_count_plot(
    df_features, top_n=10, return_fig=False
)

display(Image(salary_role_counts_path))


In [ ]:

from IPython.display import Markdown, display

def _fmt_salary(value):
    return f"{value:,.0f}".replace(",", " ") if pd.notna(value) else "N/A"

role_salary_summary = (
    df_features.groupby("primary_role", observed=True)["salary_mid_rub_capped"]
    .agg(vacancy_count_total="size", salary_median="median")
    .sort_values(by="vacancy_count_total", ascending=False)
    .head(10)
)
display(role_salary_summary)

leader = role_salary_summary.iloc[0]
leader_name = role_salary_summary.index[0]
data_block = role_salary_summary.loc[
    role_salary_summary.index.str.contains("data", case=False)
]
data_row = data_block.iloc[0] if not data_block.empty else None

data_line = (
    f"- Data-ролей в топе: {int(data_row.vacancy_count_total)} строк с медианой {_fmt_salary(data_row.salary_median)} RUB, спрос смещён в пользу analytics/BI."
    if data_row is not None
    else ""
)

lines = [
    "График: зарплаты и объём по топ-ролям.",
    f"- Лидер по объёму - {leader_name}: {int(leader.vacancy_count_total)} вакансий, медиана {_fmt_salary(leader.salary_median)} RUB.",
    data_line,
    f"- Топ-3 ролей суммарно покрывают {role_salary_summary.head(3)['vacancy_count_total'].sum()} вакансий - на них стоит фокусировать рекомендации.",
]
role_md_text = "\n".join([line for line in lines if line])
role_md = Markdown(role_md_text)
display(role_md)


In [ ]:
salary_stack_bucket_path = viz.salary_by_skills_bucket_plot(
    df_features, return_fig=False
)

display(Image(salary_stack_bucket_path))


График: зарплаты в зависимости от размера стека.
- **Рынок:** рост зарплаты коррелирует с шириной стека до 6–10 технологий, далее эффект плато.
- **Skillra:** в рекомендациях Skillra можно подсветить «оптимальный» стек для роста без избыточного перегруза.

In [ ]:

from IPython.display import Markdown, display

work_mode_series = df_features.get("work_mode", pd.Series(dtype=str)).fillna("unknown").str.lower()
work_mode_share = work_mode_series.value_counts(normalize=True)
remote_hybrid = work_mode_share.get("remote", 0) + work_mode_share.get("hybrid", 0)
office_share = work_mode_share.get("office", 0) + work_mode_share.get("field", 0)
unknown_work = work_mode_share.get("unknown", 0)

role_remote = df_features.groupby("primary_role", observed=True)["work_mode"].apply(
    lambda s: s.fillna("unknown").str.lower().value_counts(normalize=True)
)
role_counts = df_features["primary_role"].value_counts()
top_roles = role_counts.head(2).index.tolist()

def fmt_pct(value):
    return f"{float(value):.1%}" if value is not None and pd.notna(value) else "n/a"

role_lines = []
for role in top_roles:
    role_lines.append(f"{role}: {fmt_pct(role_remote.get((role, 'remote'), pd.NA))}")
remote_roles_text = "; ".join(role_lines) if role_lines else "нет ролей с достаточной выборкой"

remote_md = Markdown(
    f"""#### 3.2.2 Зарплаты по ролям и формату работы
Удалёнка/гибрид в сумме дают {remote_hybrid:.1%} вакансий, офис/field - около {office_share:.1%}; remote по топ-ролям: {remote_roles_text}. {unknown_work:.1%} work_mode = `unknown`, поэтому итоговые доли форматов стоит трактовать осторожно.
""",
)

display(remote_md)


In [ ]:

role_work_summary = eda.salary_summary_by_role_and_work_mode(df_features)
role_work_pivot = role_work_summary.pivot(index="primary_role", columns="work_mode", values="salary_median")
role_workmode_path = viz.salary_by_role_work_mode_heatmap(
    role_work_summary, return_fig=False
)

display(Image(role_workmode_path))
display(role_work_summary.head())
display(role_work_pivot)


In [ ]:

from IPython.display import Markdown, display

if "role_work_pivot" in globals():
    def _fmt_salary(value):
        return f"{value:,.0f}".replace(",", " ") if pd.notna(value) else "N/A"

    remote_cols = [c for c in role_work_pivot.columns if str(c).lower() in {"remote", "hybrid"}]
    office_cols = [c for c in role_work_pivot.columns if str(c).lower() in {"office", "field"}]

    remote_best = (
        role_work_pivot[remote_cols].stack().sort_values(ascending=False) if remote_cols else pd.Series(dtype=float)
    )
    office_best = (
        role_work_pivot[office_cols].stack().sort_values(ascending=False) if office_cols else pd.Series(dtype=float)
    )

    remote_role, remote_mode, remote_salary = ("unknown", "remote", pd.NA)
    if not remote_best.empty:
        (remote_role, remote_mode), remote_salary = remote_best.index[0], remote_best.iloc[0]

    office_role, office_mode, office_salary = ("unknown", "office", pd.NA)
    if not office_best.empty:
        (office_role, office_mode), office_salary = office_best.index[0], office_best.iloc[0]

    work_mode_series = df_features.get("work_mode", pd.Series(dtype=str)).fillna("unknown").str.lower()
    work_mode_counts = work_mode_series.value_counts()
    total_vacancies = work_mode_counts.sum()
    remote_hybrid_share = (
        work_mode_counts.get("remote", 0) + work_mode_counts.get("hybrid", 0)
    ) / total_vacancies if total_vacancies else 0

    conclusion_md = Markdown(
        f"*Ключ: премии за формат (автоизвлечение).*\n\n"
        f"Remote/hybrid {remote_role} - {_fmt_salary(remote_salary)} ₽ медиана ({remote_mode}); "
        f"офисный/field пик: {office_role} - {_fmt_salary(office_salary)} ₽ ({office_mode}).\n"
        f"Доля remote+hybrid в срезе: {remote_hybrid_share:.1%}."
    )
    display(conclusion_md)


*График: зарплаты по ролям и формату работы.*
- **Выводы про рынок:** гибрид у продуктовых и аналитических ролей сопоставим с офисом; backend/ML на полном remote часто имеют дисконт.
- **Что это даёт Skillra:** при подборе ролей подсвечиваем, сколько можно ожидать на remote/office и стоит ли искать гибрид ради зарплатной премии.

In [ ]:

from IPython.display import Markdown, display

if 'fmt_pct' not in globals():
    def fmt_pct(value):
        return f"{float(value):.1%}" if value is not None and pd.notna(value) else "n/a"

remote_role_lines = []
for role in top_roles:
    remote_role_lines.append(f"{role}: {fmt_pct(role_remote.get((role, 'remote'), pd.NA))}")
remote_role_text = '; '.join(remote_role_lines) if remote_role_lines else 'нет явных лидеров по remote'

remote_md = Markdown(
    f"""#### 3.2.3 Доля удалёнки и junior-friendly по ролям

Remote+hybrid форматы занимают {remote_hybrid:.1%} выборки против {office_share:.1%} офиса/field; {unknown_work:.1%} строк имеют `unknown` work_mode. Из-за доли unknown оценки удалёнки/гибрида - ориентиры, не финальные выводы.
Top remote-доли: {remote_role_text}.

Поля work_mode с низким coverage учитываем только как дополнительный сигнал, не как жёсткий сегмент."""
)

display(remote_md)


In [ ]:

remote_share = eda.remote_share_by_role(df_features)
junior_roles = eda.junior_friendly_share(df_features, group_col="primary_role")
remote_share_path = viz.remote_share_by_role_bar(remote_share, return_fig=False)

display(Image(remote_share_path))
display(remote_share.head())
display(junior_roles.head())


In [ ]:

work_mode_share = df_features["work_mode"].value_counts(normalize=True, dropna=False)
remote_hybrid_share = work_mode_share.get("remote", 0) + work_mode_share.get("hybrid", 0)
unknown_work_mode_share = work_mode_share.get("unknown", 0)

display(work_mode_share)
display(pd.Series({"remote_or_hybrid_share": remote_hybrid_share, "unknown_work_mode_share": unknown_work_mode_share}))


In [ ]:
from IPython.display import Markdown, display

remote_hybrid_share = work_mode_share.get("remote", 0) + work_mode_share.get("hybrid", 0)
unknown_work_mode_share = work_mode_share.get("unknown", 0)
work_mode_md = Markdown(
    f"**Форматы работы:** remote+hybrid - {remote_hybrid_share:.1%}, офис - {work_mode_share.get('office', 0):.1%}, field - {work_mode_share.get('field', 0):.1%}, unknown - {unknown_work_mode_share:.1%}. "
    "Высокая доля unknown (треть выборки) смещает выводы по форматам."
)
display(work_mode_md)


### 3.3 Топ работодатели, бенефиты и soft-skills
Определяем лидеров по количеству вакансий и проверяем, какие бенефиты и мягкие навыки они чаще всего упоминают.

In [ ]:
top_employers_df = eda.top_employers(df_features)

top_employers_df.head(10)

In [ ]:
benefits_path = viz.benefits_employer_heatmap(
    df_features,
    top_n_employers=10,
    return_fig=False,
)

display(Image(benefits_path))


График: бенефиты у топ-работодателей.
- **Рынок:** ДМС, релокация и обучение чаще у крупных работодателей.
- **Skillra:** можно рекомендовать компании с подходящими пакетами и объяснять, где искать ДМС/релокацию.

In [ ]:
soft_path = viz.soft_skills_employer_heatmap(
    df_features,
    top_n_employers=10,
    return_fig=False,
)

display(Image(soft_path))


График: soft skills у топ-работодателей.
- **Рынок:** коммуникация, аналитическое мышление и командная работа встречаются чаще всего.
- **Skillra:** объясняем пользователям, какие soft skills нужны для входа в топ-компании.

#### 3.3.1 Топ навыков для data-ролей

In [ ]:
skill_cols = eda.hard_skill_columns(df_features)
top_skills_path = viz.top_skills_bar(
    df_features, skill_cols=skill_cols, role_filter="data", top_n=12, return_fig=False
)

def humanize_skill(skill: str) -> str:
    if hasattr(viz, "_humanize_skill_name"):
        return viz._humanize_skill_name(skill)
    return skill.replace("skill_", "").replace("has_", "").replace("_", " ").strip()

display(Image(top_skills_path))


In [ ]:

data_roles_mask = df_features["primary_role"].str.contains("data", case=False, na=False)
data_skill_freq = df_features.loc[data_roles_mask, skill_cols].sum().sort_values(ascending=False).head(10)
data_role_count = int(data_roles_mask.sum())

display(pd.Series({"data_role_count": data_role_count}))
display(data_skill_freq)


In [ ]:
data_top = data_skill_freq.head(5)

skill_lines = []
if data_role_count > 0:
    skill_lines = [
        f"{humanize_skill(skill)} - {count / data_role_count:.1%} (n={int(count)})"
        for skill, count in data_top.items()
    ]

if not skill_lines:
    skill_lines = ["Нет наблюдений для data-ролей - топ навыков не определён."]

skill_md = Markdown(
    f"""
Data-вакансий: {data_role_count}.
Топ-5 хард-скиллов (доля строк):
{'<br>'.join(skill_lines)}
Высокая доля unknown work_mode={work_mode_share.get('unknown', 0):.1%} может смещать оценки сегмента.
"""
)
display(skill_md)


In [ ]:
top3_skills = [humanize_skill(skill) for skill in data_top.index[:3]]
if top3_skills:
    insight_text = (
        f"*Data-рынок держится на базе {', '.join(top3_skills)}: они появляются чаще всего и должны идти первыми в рекомендациях Skillra.*"
    )
else:
    insight_text = "*Data-рынок: недостаточно данных для авто-вывода по топ-навыкам.*"

insight_md = Markdown(insight_text)
display(insight_md)


In [ ]:

from IPython.display import Markdown, display

if '_fmt_salary' not in globals():
    def _fmt_salary(value):
        return f"{value:,.0f}".replace(",", " ") if pd.notna(value) else "N/A"

top10_share = top_employers_df.head(10)["share"].sum()
leader = top_employers_df.head(1).iloc[0]
leader_share = leader.get("share", pd.NA)
leader_salary = leader.get("salary_median", pd.NA)
leader_name = leader.get("employer") if isinstance(leader, dict) or "employer" in leader else getattr(leader, "name", "N/A")

employers_md = Markdown(
    f"""#### Выводы по работодателям
- Топ-10 компаний покрывают {top10_share:.1%} выборки: лидер ({leader_name}) даёт {leader_share:.1%} вакансий с медианой {_fmt_salary(leader_salary)} ₽ - важно смотреть на стек и роль.
- Часто встречающиеся бенефиты: ДМС, гибкий график, возможность удалёнки/релокации.
- В soft-skills топе - коммуникация и самоорганизация; на них стоит делать акцент при подготовке кандидатов.
"""
)

display(employers_md)


### 3.4 Английский и образование
Смотрим требования к языку и образованию: распределения долей, премии по зарплатам и их интерпретация.

In [ ]:

req_df = df_features.copy()
for col in ["lang_english_required", "edu_required", "edu_technical"]:
    if col in req_df:
        req_df[col] = req_df[col].fillna(False)

english_stats = eda.english_requirement_stats(req_df)
education_stats = eda.education_requirement_stats(req_df)
english_unknown_share = english_stats.loc[english_stats['english_level'] == 'unknown', 'share'].sum()
education_unknown_share = 1 - education_stats["share"].sum()

display(english_stats)
display(education_stats)
display(pd.Series({
    "english_unknown_share": english_unknown_share,
    "education_unknown_share": education_unknown_share,
}))


In [ ]:
salary_english_path = viz.salary_by_english_level_plot(req_df, return_fig=False)

display(Image(salary_english_path))


Короткое пояснение: «Английский не требуется» - это явное отсутствие требования, а «Требование не указано» - когда работодатель не уточнил уровень языка.

In [ ]:

from IPython.display import Markdown, display

english_stats_sorted = english_stats.sort_values(by="share", ascending=False)
display(english_stats_sorted)

b2_plus_share = english_stats.loc[
    english_stats["english_level"].isin(["b2", "c1_plus", "c1", "c2"]),
    "share",
].sum()
explicit_share = 1 - english_unknown_share
b2_salary = english_stats.loc[english_stats["english_level"] == "b2", "salary_median"].max()

english_md = Markdown(
    f"""График: зарплаты по требуемому уровню английского.

- Явное указание уровня встречается в {explicit_share:.1%} строк; B2+ всего {b2_plus_share:.1%}, медиана {_fmt_salary(b2_salary)} RUB.
- {english_unknown_share:.1%} вакансий без уровня → выводы по премии трактуем осторожно."""
)
display(english_md)


In [ ]:
salary_edu_path = viz.salary_by_education_level_plot(req_df, return_fig=False)

display(Image(salary_edu_path))


In [ ]:

from IPython.display import Markdown, display

education_stats_sorted = education_stats.sort_values(by="vacancy_count", ascending=False)
display(education_stats_sorted)

edu_known_share = 1 - education_unknown_share if education_unknown_share is not None else education_stats_sorted["share"].sum()
edu_leader = education_stats_sorted.iloc[0]
tech_share = education_stats_sorted.loc[
    education_stats_sorted["education_level"].str.contains("technical", case=False, na=False),
    "share",
].sum()

education_md = Markdown(
    f"""График: зарплаты по требованиям к образованию.

- Чёткие требования указаны в {edu_known_share:.1%} вакансий; лидер по объёму - {edu_leader.education_level} ({edu_leader.share:.1%}, медиана {_fmt_salary(edu_leader.salary_median)} RUB).
- Техническое образование встречается в {tech_share:.1%} строк, сильной зарплатной премии не видно - фокус на навыках."""
)
display(education_md)


In [ ]:
from IPython.display import Markdown, display

english_unknown_share = english_stats.loc[english_stats['english_level'] == 'unknown', 'share'].sum()
education_unknown_share = 1 - education_stats["share"].sum()
no_english_share = english_stats.loc[english_stats['english_level'] == 'no_english', 'share'].iat[0]
b2plus_share = english_stats.loc[english_stats['english_level'].isin(['b2', 'c1_plus'])]['share'].sum()
no_degree_share = education_stats.loc[education_stats['education_level'] == 'no_degree_required', 'share'].iat[0]
tech_degree_share = education_stats.loc[education_stats['education_level'] == 'technical_only', 'share'].iat[0]
any_degree_share = education_stats.loc[education_stats['education_level'] == 'any_degree', 'share'].iat[0]

req_md = Markdown(f"""
#### Выводы по требованиям
- Английский не требуется в {no_english_share:.1%} вакансий; уровни B2+ встречаются в {b2plus_share:.1%}. Unknown уровень - {english_unknown_share:.1%}, поэтому премии нужно трактовать аккуратно.
- Образование: без диплома {no_degree_share:.1%}, техническое высшее {tech_degree_share:.1%}, любое высшее {any_degree_share:.1%}; неизвестно - {education_unknown_share:.1%}.
- Эти требования сегментируют рынок, их стоит учитывать в продуктовых фильтрах.
""")
display(req_md)


### 3.5 Корреляционный анализ числовых признаков
Используем готовые функции `eda.correlation_matrix` и `viz.corr_heatmap`, чтобы понять связи между зарплатой и числовыми фичами.

In [ ]:
corr_cols = [
    col
    for col in [
        "salary_from",
        "salary_to",
        "vacancy_age_days",
        "tech_stack_size",
        "core_data_skills_count",
        "ml_stack_count",
        "description_len_words",
    ]
    if col in df_features.columns
]
correlations = eda.correlation_matrix(df_features, cols=corr_cols)
corr_fig_path = viz.corr_heatmap(correlations, return_fig=False)

display(Image(corr_fig_path))


*Связь с зарплатой:* `salary_from` и `salary_to` ожидаемо коррелируют между собой, негативная связь с `vacancy_age_days` подчёркивает, что свежие вакансии быстрее набирают отклики.

*Выводы для рынка:* размер стека и длина описания умеренно положительно связаны с зарплатами.

*Skillra:* можно использовать эти связи в подсказках по улучшению резюме и в приоритизации вакансий для пользователей.

In [ ]:

from IPython.display import Markdown, display

if '_fmt_salary' not in globals():
    def _fmt_salary(value):
        return f"{value:,.0f}".replace(",", " ") if pd.notna(value) else "N/A"

city_tier_share = df_features.get("city_tier", pd.Series(dtype=str)).value_counts(normalize=True)
moscow_share = city_tier_share.get("Moscow", 0) + city_tier_share.get("SPb", 0)
remote_hybrid = work_mode_share.get("remote", 0) + work_mode_share.get("hybrid", 0)
unknown_work = work_mode_share.get("unknown", 0)

domain_share = domain_salary.set_index('domain')['share'] if 'domain' in domain_salary else pd.Series(dtype=float)
fintech_share = domain_share.get('fintech', pd.NA)
product_share = domain_share.get('product_it', pd.NA)
english_required = 1 - english_unknown_share if english_unknown_share is not None and not pd.isna(english_unknown_share) else pd.NA

insights_md = Markdown(
    f"""### 3.6 Инсайты EDA
- **Что мы сделали:** построили тепловые карты по зарплатам (грейд×город, роль×формат), разложили долю удалёнки, выгрузили топ работодателей/бенефитов/soft-skills, посмотрели корреляции.
- **Почему именно так:** инвесторам и пользователям важны премии по локации/формату и качество работодателей; корреляции показывают, какие числовые признаки двигают зарплату.
- **Основные выводы про рынок:** Москва/СПб дают {moscow_share:.1%} вакансий и премию по senior до {_fmt_salary(senior_moscow)}₽; удалёнка/гибрид присутствует в {remote_hybrid:.1%} вакансий при {unknown_work:.1%} `unknown` work_mode; финтех {fintech_share:.1%} и продуктовые IT {product_share:.1%} держат премию, английский/образование явно требуют ~{english_required:.1%} выборки.
- **Что это даёт продукту Skillra:** формируем подсказки по целевым кластерам (например, data-ролям в крупных городах или remote fintech) и готовим аргументы для карьерного трекинга.
- **Мини-чек-лист:** EDA проходит на датасете {fmt_int(len(df_features))}×{fmt_int(df_features.shape[1])} без изменения формы; собрали числовые сводки по зарплатам/форматам/ролям, долям remote и junior-friendly, топам работодателей и навыков.
""",
)

bridge_md = Markdown("Завершаем аналитическую часть и переходим к блоку визуализаций и продуктовым персонам.")
display(insights_md, bridge_md)


## Этап 4. Визуализации и сводка
Используем готовые функции из `viz.py`, сохраняем графики в `reports/figures`. Графики покрывают зарплаты, форматы работы, навыковые тепловые карты, soft-skills и распределения.

In [ ]:

role_flags = ["backend", "frontend", "fullstack", "data", "analyst", "product"]
salary_role_path = viz.salary_by_role_flags_box(
    df_features,
    roles=role_flags,
    salary_col="salary_mid_rub_capped",
    return_fig=False,
)

display(Image(salary_role_path))


Блок визуализаций: распределение зарплат по ролям (роли с достаточной выборкой или топ по данным).
- **Рынок:** медианы заметно различаются между популярными ролями, при этом разброс внутри каждой остаётся высоким.
- **Skillra:** график помогает кандидатам соотнести ожидания по зарплате с реальными вилками внутри своей роли.

In [ ]:
workmode_path = viz.work_mode_share_by_city(df_features, return_fig=False)

display(Image(workmode_path))


In [ ]:
from IPython.display import Markdown, display

city_work_mode = (
    df_features.assign(work_mode=df_features.get("work_mode", pd.Series(dtype=str)).fillna("unknown").str.lower())
    .groupby(["city_tier", "work_mode"], observed=True)
    .size()
    .unstack(fill_value=0)
)
city_mode_share = city_work_mode.div(city_work_mode.sum(axis=1), axis=0)
unknown_city = city_mode_share.get("unknown", pd.Series(dtype=float))
remote_city = city_mode_share.get("remote", pd.Series(dtype=float))

remote_top_city = remote_city.sort_values(ascending=False).head(1) if not remote_city.empty else pd.Series(dtype=float)
unknown_top_city = unknown_city.sort_values(ascending=False).head(1) if not unknown_city.empty else pd.Series(dtype=float)

remote_line = (
    f"максимальная доля удалёнки - {remote_top_city.index[0]} {remote_top_city.iloc[0]:.1%}"
    if not remote_top_city.empty
    else "удалёнка не заявлена"
)
unknown_line = (
    f"максимальный unknown - {unknown_top_city.index[0]} {unknown_top_city.iloc[0]:.1%}"
    if not unknown_top_city.empty
    else "unknown не заявлен"
)

city_md = Markdown(f"*Форматы по городам (авто-вывод):* {remote_line}; {unknown_line}.")
display(city_md)


Блок визуализаций: формат работы по типу города.
- **Рынок:** удалёнка концентрируется в столицах, в регионах преобладает офис/гибрид.
- **Skillra:** можно подсказывать форматы работы и релокацию/удалёнку в зависимости от города пользователя.

In [ ]:
skill_heatmap_path = viz.heatmap_skills_by_grade(skill_share, return_fig=False)

display(Image(skill_heatmap_path))


In [ ]:
from IPython.display import Markdown, display

if "skill_share" in globals():
    def _human(skill: str) -> str:
        if hasattr(viz, "_humanize_skill_name"):
            return viz._humanize_skill_name(skill)
        return skill.replace("skill_", "").replace("has_", "").replace("_", " ").strip()

    grade_top_skills = {}
    for grade in ["intern", "junior", "middle", "senior", "lead"]:
        if grade in skill_share.columns:
            top_skill = skill_share[grade].dropna().sort_values(ascending=False).head(1)
            if not top_skill.empty:
                grade_top_skills[grade] = (_human(top_skill.index[0]), top_skill.iloc[0])
            else:
                grade_top_skills[grade] = (None, None)
        else:
            grade_top_skills[grade] = (None, None)

    lines = []
    for grade, (skill, share) in grade_top_skills.items():
        if skill is not None and share is not None:
            lines.append(f"{grade} - {skill} ({share:.1%})")
        else:
            lines.append(f"{grade} - недостаточно данных")

    skills_md = Markdown(
        "*Навыки по грейдам:* "
        + "; ".join(lines)
        + "."
    )
    display(skills_md)


Блок визуализаций: навыки по грейдам.
- **Рынок:** у middle/senior шире стек и выше доля продвинутых skills.
- **Skillra:** подсвечиваем пользователю, какие навыки добавлять, чтобы перейти в следующий грейд.

### 4.1 Выводы по графикам
- **Что мы сделали:** собрали набор готовых графиков (боксплоты, bar+count, тепловые карты) прямо из кода `viz.py` и встроили в ноутбук.
- **Почему именно так:** хотим, чтобы ноутбук был self-contained и понятен инвестору: каждый график сразу отображается и сохраняется в `reports/figures`.
- **Основные выводы про рынок:** медианы и объёмы видны на одном экране, тепловые карты подчёркивают премии по локации/формату.
- **Что это даёт продукту Skillra:** готовы reusable-компоненты для веб-интерфейса (дашборд рынка) и презентаций.
- **Мини-чек-лист:** исходный датасет 7 026×164 остаётся неизменным, добавлены готовые артефакты в `reports/figures` (зарплаты по ролям, формат работы, heatmap навыков).
- **К какому следующему шагу это подводит:** переходим к персонализации (персоны) и масштабированию данных.

## Продуктовая часть: персоны и skill-gap
Персоны из контекста Skillra: студент (Junior DA/DS), свитчер в BI/продукт, middle аналитик. Используем `personas.analyze_persona` как основной API: он возвращает market summary, skill-gap и рекомендованные навыки.

In [ ]:
personas_mod = importlib.reload(personas)
persona_objects = [
    personas_mod.DATA_STUDENT,
    personas_mod.SWITCHER_BI,
    personas_mod.MID_DATA_ANALYST,
]

persona_reports = {}
for p in persona_objects:
    analysis = personas_mod.analyze_persona(df_features, p, top_k=10)
    persona_reports[p.name] = analysis

    persona_profile = pd.DataFrame({
        "persona": [p.name],
        "description": [p.description],
        "goals": [", ".join(p.goals) if p.goals else "-"],
        "limitations": [", ".join(p.limitations) if p.limitations else "-"],
        "target_role": [p.target_role],
        "target_grade": [p.target_grade],
        "target_city_tier": [p.target_city_tier],
        "target_work_mode": [p.target_work_mode],
    })
    display(persona_profile)

    summary_df = pd.DataFrame([analysis["market_summary"]])
    display(summary_df)

    top_demand = analysis.get("top_skill_demand")
    if isinstance(top_demand, pd.DataFrame) and not top_demand.empty:
        display(top_demand.head(5))

    display(analysis["skill_gap"].head(10))
    top_reco = analysis["recommended_skills"][:5]
    print(f"Рекомендуем добрать: {', '.join(top_reco) if top_reco else 'нет явных гэпов'}")
    print('-' * 40)


In [ ]:
gap_student = persona_reports.get("data_student", {}).get("skill_gap")
path_gap_student = personas_mod.plot_persona_skill_gap(
    gap_student, personas_mod.DATA_STUDENT, return_fig=False
)

display(Image(path_gap_student))


In [ ]:
from IPython.display import Markdown, display

student_report = persona_reports.get("data_student", {})
student_gap = student_report.get("skill_gap", pd.DataFrame()).head(3)
student_market = student_report.get("market_summary", {})
student_skills = [row.skill_name for row in student_gap.itertuples()] if not student_gap.empty else []
student_md = Markdown(
    f"Skill-gap для персоны *data_student* (n={student_market.get('vacancy_count_total', student_market.get('vacancy_count', 0))}): "
    + (", ".join(student_skills) if student_skills else "нет выраженных гэпов")
    + ". Remote доля - "
    + (f"{student_market.get('remote_share', 0):.1%}" if 'remote_share' in student_market else "-")
    + ", junior-friendly - "
    + (f"{student_market.get('junior_friendly_share', 0):.1%}" if 'junior_friendly_share' in student_market else "-")
    + "."
)
display(student_md)


In [ ]:
gap_switcher = persona_reports.get("switcher_bi", {}).get("skill_gap")
path_gap_switcher = personas_mod.plot_persona_skill_gap(
    gap_switcher, personas_mod.SWITCHER_BI, return_fig=False
)

display(Image(path_gap_switcher))


In [ ]:
from IPython.display import Markdown, display

bi_report = persona_reports.get("switcher_bi", {})
bi_gap = bi_report.get("skill_gap", pd.DataFrame()).head(3)
bi_market = bi_report.get("market_summary", {})
bi_skills = [row.skill_name for row in bi_gap.itertuples()] if not bi_gap.empty else []
bi_md = Markdown(
    f"Skill-gap для персоны *switcher_bi* (n={bi_market.get('vacancy_count_total', bi_market.get('vacancy_count', 0))}): "
    + (", ".join(bi_skills) if bi_skills else "нет выраженных гэпов")
    + f". Remote+hybrid - {bi_market.get('remote_share', 0):.1%}, junior-friendly - {bi_market.get('junior_friendly_share', 0):.1%}."
)
display(bi_md)


*Визуализация skill-gap даёт готовый виджет для личного кабинета Skillra: пользователь видит, какие навыки закрывают 
максимальную долю вакансий в целевом сегменте, и может кликнуть в рекомендации курсов/проектов.*

In [ ]:
from IPython.display import Markdown, display

student_n = persona_reports.get("data_student", {}).get("market_summary", {}).get("vacancy_count_total", persona_reports.get("data_student", {}).get("market_summary", {}).get("vacancy_count", 0))
switcher_n = persona_reports.get("switcher_bi", {}).get("market_summary", {}).get("vacancy_count_total", persona_reports.get("switcher_bi", {}).get("market_summary", {}).get("vacancy_count", 0))
mid_n = persona_reports.get("mid_data_analyst", {}).get("market_summary", {}).get("vacancy_count_total", persona_reports.get("mid_data_analyst", {}).get("market_summary", {}).get("vacancy_count", 0))
summary_md = Markdown(f"""
### Выводы по персонам
- Студент DA/DS: сегмент n={student_n}, ключевые гэпы - топ-3 навыка из графика; remote {persona_reports['data_student']['market_summary'].get('remote_share', 0):.1%}.
- Свитчер BI: n={switcher_n}, приоритеты из графика; remote {persona_reports['switcher_bi']['market_summary'].get('remote_share', 0):.1%}.
- Mid аналитик: n={mid_n}, опираемся на видимые гэпы; выводы делаем только по отображённым навыкам.
""")
display(summary_md)


In [ ]:
from IPython.display import Markdown, display

notes_md = Markdown("""
### Пояснения по персонам
- Все выводы основаны на графике skill-gap: обсуждаем только навыки, которые реально в топе гэпов.
- n сегментов явно указано; при уменьшении выборки ниже min_market_n анализ не проводим.
- Высокая доля unknown по грейдам/форматам учитывается при интерпретации.
""")
display(notes_md)


## Итоговые выводы и чек-лист
### Выводы про рынок
- Зарплаты растут по грейду и укрупнённости города; в Москве медиана senior ~275k против 178k в регионах, удалёнка концентрируется в ролях data/ML (до 81%).
- Доменные лидеры (финтех ~17%, продуктовые IT ~12%) и топ работодатели держат медианы выше среднего, но 54% вакансий без домена - учитываем смещение.
- Требования к английскому/образованию редко жёсткие (только ~15–18% выборки с явными требованиями), поэтому рынок открыт для широкого пула кандидатов.
- Навыковая карта подтверждает спрос на SQL/BI для junior и продуктовых аналитиков, ML/infra - для senior; это отражено в skill-gap персонам.
- Корреляции показывают влияние размера стека и насыщенности описания на уровень вилки, что можно использовать в скоринге вакансий.

### Выводы для Skillra
- Готовые витрины (`hh_clean`, `hh_features`, `market_view`) позволяют строить карьерные рекомендации без ручной очистки.
- API `analyze_persona` даёт готовый блок для пользовательских сценариев (оценка сегмента, gap, next best skill).
- Самые ценные сегменты для продукта - data/ML и аналитика с высоким спросом на удалёнку и junior-friendly роли.
- Карта бенефитов/soft-skills по работодателям помогает подбирать целевые компании под цели пользователя.

### Чек-лист выполнения ТЗ
| Этап | Что сделано | Секция ноутбука |
| --- | --- | --- |
| 0. Данные | Парсер HH, raw head/info, профиль пропусков | Этап 0 |
| 1. Предобработка | Пропуски/булевы/дубликаты, `missing_share` | Этап 1 |
| 2. Признаки | `city_tier`, `work_mode`, стек, тексты, junior-friendly | Этап 2 |
| 3. EDA | Зарплаты, форматы, роли, навыки, домены, язык, образование, работодатели, корреляции | Этап 3 |
| 4. Визуализация | Графики зарплат/навыков/soft-skills/распределений | Этап 4 |
| Продуктовый слой | `analyze_persona`, skill-gap, рекомендации | Персоны |

### Проверка формата ноутбука
Чтобы убедиться в валидности после изменений, выполните:
```bash
python -c "import nbformat; nbformat.read('notebooks/01_hse_project.ipynb', as_version=4); print('OK')"
```